In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [Path('titanic.csv'), Path('LAB1') / 'titanic.csv', Path('LAB1') / 'Bai2_Trang15-17' / 'titanic.csv', Path('../titanic.csv'), Path('/content/titanic.csv'), Path('/mnt/data/titanic.csv')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    try:
        from google.colab import files
    except ImportError:
        files = None
    if files is not None:
        print('Chưa tìm thấy titanic.csv. Hãy chọn tệp CSV để tải lên Colab.')
        uploaded = files.upload()
        uploaded_path = next((Path(name) for name in uploaded if Path(name).name.lower() == 'titanic.csv'), None)
        if uploaded_path is not None and uploaded_path.is_file():
            data_path = uploaded_path
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy titanic.csv. Thư mục chạy hiện tại: {Path.cwd()}. Hãy tải CSV lên Colab hoặc đặt CSV trong thư mục này rồi chạy lại ô.')
df = pd.read_csv(data_path)
plot_data = df.dropna(subset=['embarked', 'pclass', 'sex', 'survived']).copy()
summary = (plot_data.groupby(['sex', 'embarked', 'pclass'], observed=True)['survived']
           .agg(ty_le_song='mean', so_nguoi='size').reset_index())
print('Tỷ lệ sống và cỡ mẫu theo giới tính, cảng và hạng vé:')
display(summary)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, sex in zip(axes, ['female', 'male']):
    subset = summary[summary['sex'] == sex]
    rates = subset.pivot(index='embarked', columns='pclass', values='ty_le_song').reindex(index=['S', 'C', 'Q'], columns=[1, 2, 3])
    rates.plot(kind='bar', ax=ax, ylim=(0, 1), color=['#4c78a8', '#f2a541', '#59a14f'])
    ax.set_title(f'Giới tính: {sex}')
    ax.set_xlabel('Cảng lên tàu (S, C, Q)')
    ax.set_ylabel('Tỷ lệ sống sót')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(title='Hạng vé')
    ax.tick_params(axis='x', rotation=0)
fig.suptitle('Tỷ lệ sống sót theo cảng và hạng vé, phân biệt giới tính')
plt.tight_layout()
plt.show()

print('Nhận xét: so sánh tỷ lệ giữa các hạng vé trong từng giới tính và cảng; hãy xem cỡ mẫu ở bảng để tránh kết luận từ nhóm quá nhỏ. Các nhóm thiếu quan sát sẽ không có cột trên biểu đồ.')